In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

import warnings
warnings.filterwarnings("ignore")

In [3]:
!pip install dagshub mlflow
import dagshub
dagshub.init(repo_owner='smama23', repo_name='MLassignment2', mlflow=True)

import mlflow
mlflow.set_experiment("LogisticRegression_Training")
with mlflow.start_run():
  mlflow.log_param('parameter name', 'value')
  mlflow.log_metric('metric name', 1)

Accessing as smama23

Initialized MLflow to track repo "smama23/MLassignment2"

Repository smama23/MLassignment2 initialized!

🏃 View run melodic-elk-273 at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/0/runs/83cf317842564b66bdbe00fc14966f71
🧪 View experiment at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/0


In [4]:
train_transaction = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv")
train_identity = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv")

df = train_transaction.merge(train_identity, on="TransactionID", how="left")

print(df.shape)
df.head()

(590540, 434)


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M


## CLEANING

In [5]:
df_clean = df.copy()

y = df_clean["isFraud"]
df_clean = df_clean.drop(columns=["isFraud"])

df_clean = df_clean.fillna(-999)

print("Missing values after cleaning:", df_clean.isnull().sum().sum())

Missing values after cleaning: 0


In [6]:
with mlflow.start_run(run_name="LogisticRegression_Cleaning_v1"):
    
    df_clean = df.copy()
    
    y = df_clean["isFraud"]
    df_clean = df_clean.drop(columns=["isFraud"])
    
    df_clean = df_clean.fillna(-999)
    
    mlflow.log_param("cleaning_method", "fillna_-999")
    mlflow.log_metric("num_missing_after", df_clean.isnull().sum().sum())
    mlflow.log_metric("num_features", df_clean.shape[1])

🏃 View run LogisticRegression_Cleaning_v1 at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/0/runs/1ef13a5d95c844e283a8c8aa5d4b53ec
🧪 View experiment at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/0


## FEATURE ENGINEERING

In [7]:
df_fe = df_clean.copy()

df_fe["hour"] = (df_fe["TransactionDT"] // 3600) % 24
df_fe["day"] = (df_fe["TransactionDT"] // (3600 * 24)) % 7

df_fe["log_amount"] = np.log1p(df_fe["TransactionAmt"])

In [8]:
def frequency_encoding(df, cols):
    df = df.copy()
    for col in cols:
        freq = df[col].value_counts() / len(df)
        df[col] = df[col].map(freq)
    return df

df_enc = df_fe.copy()

cat_cols = df_enc.select_dtypes(include="object").columns

df_enc = frequency_encoding(df_enc, cat_cols)

In [9]:
with mlflow.start_run(run_name="LogisticRegression_FeatureEngineering_v2"):
    
    df_fe = df_clean.copy()
    
    df_fe["hour"] = df_fe["TransactionDT"]
    df_fe["day"] = df_fe["TransactionDT"]
    df_fe["log_amount"] = np.log1p(df_fe["TransactionAmt"])
    
    mlflow.log_param("features_added", "hour, day, log_amount")
    mlflow.log_metric("num_features_after_fe", df_fe.shape[1])

🏃 View run LogisticRegression_FeatureEngineering_v2 at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/0/runs/3b9023b9c153464597b5cab1015ae37d
🧪 View experiment at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/0


## FEATURE SELECTION

In [10]:
df_fs = df_enc.copy()

nunique = df_fs.nunique()
cols_to_keep = nunique[nunique > 1].index

df_fs = df_fs[cols_to_keep]

In [11]:
from sklearn.preprocessing import LabelEncoder

with mlflow.start_run(run_name="LogisticRegression_FeatureSelection_v1"):
    
    df_enc = df_fe.copy()
    
    cat_cols = df_enc.select_dtypes(include="object").columns
    
    for col in cat_cols:
        le = LabelEncoder()
        df_enc[col] = le.fit_transform(df_enc[col].astype(str))
    
    nunique = df_enc.nunique()
    cols_to_keep = nunique[nunique > 1].index
    df_fs = df_enc[cols_to_keep]
    
    mlflow.log_param("encoding", "label_encoding")
    mlflow.log_param("feature_selection", "remove_constant")
    mlflow.log_metric("num_features_final", df_fs.shape[1])

🏃 View run LogisticRegression_FeatureSelection_v1 at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/0/runs/b3b0f7caee4c4d489fc1cc0d8bb561ad
🧪 View experiment at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/0


## TRAINING

In [12]:
X_train, X_val, y_train, y_val = train_test_split(
    df_fs, y, test_size=0.2, random_state=42, stratify=y
)

In [13]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

In [14]:
model = LogisticRegression(
    C=0.1,
    class_weight="balanced",
    max_iter=1000
)

model.fit(X_train_scaled, y_train)

LogisticRegression(C=0.1, class_weight='balanced', max_iter=1000)

In [15]:
y_train_pred = model.predict_proba(X_train_scaled)[:, 1]
y_val_pred = model.predict_proba(X_val_scaled)[:, 1]

train_auc = roc_auc_score(y_train, y_train_pred)
val_auc = roc_auc_score(y_val, y_val_pred)

print(f"Train AUC: {train_auc:.4f}")
print(f"Validation AUC: {val_auc:.4f}")

Train AUC: 0.8505
Validation AUC: 0.8486


In [16]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

with mlflow.start_run(run_name="LogisticRegression_Training_v2_improved"):
    
    df_enc = df_fe.copy()
    cat_cols = df_enc.select_dtypes(include="object").columns
    df_enc = frequency_encoding(df_enc, cat_cols)
    
    nunique = df_enc.nunique()
    df_fs = df_enc[nunique[nunique > 1].index]
    
    X_train, X_val, y_train, y_val = train_test_split(
        df_fs, y, test_size=0.2, random_state=42, stratify=y
    )
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    
    model = LogisticRegression(
        C=0.1,
        class_weight="balanced",
        max_iter=1000
    )
    
    model.fit(X_train_scaled, y_train)
    
    y_train_pred = model.predict_proba(X_train_scaled)[:, 1]
    y_val_pred = model.predict_proba(X_val_scaled)[:, 1]
    
    train_auc = roc_auc_score(y_train, y_train_pred)
    val_auc = roc_auc_score(y_val, y_val_pred)
    
    mlflow.log_param("encoding", "frequency")
    mlflow.log_param("C", 0.1)
    mlflow.log_param("class_weight", "balanced")
    
    mlflow.log_metric("train_auc", train_auc)
    mlflow.log_metric("val_auc", val_auc)
    
    mlflow.sklearn.log_model(model, "model")
    
    print("Improved Train AUC:", train_auc)
    print("Improved Val AUC:", val_auc)

2026/05/03 14:01:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/03 14:01:32 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Improved Train AUC: 0.8525978828252598
Improved Val AUC: 0.850039059359219
🏃 View run LogisticRegression_Training_v2_improved at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/0/runs/b78a5740514c48359a9abf20ac7d4ff5
🧪 View experiment at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/0


In [17]:
from sklearn.base import BaseEstimator, TransformerMixin

class FrequencyEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, cols):
        self.cols = cols
        self.freq_maps = {}

    def fit(self, X, y=None):
        for col in self.cols:
            freq = X[col].value_counts() / len(X)
            self.freq_maps[col] = freq
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.cols:
            X[col] = X[col].map(self.freq_maps[col]).fillna(0)
        return X


class FeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X["hour"] = X["TransactionDT"]
        X["day"] = X["TransactionDT"]
        X["log_amount"] = np.log1p(X["TransactionAmt"])
        return X


class FillNaTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, fill_value=-999):
        self.fill_value = fill_value

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X.fillna(self.fill_value)

In [18]:
from sklearn.pipeline import Pipeline

cat_cols = df.select_dtypes(include="object").columns

pipeline = Pipeline([
    ("cleaning", FillNaTransformer()),  # 🔥 FIX
    ("feature_engineering", FeatureEngineer()),
    ("frequency_encoding", FrequencyEncoder(cat_cols)),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        C=0.1,
        class_weight="balanced",
        max_iter=1000
    ))
])

In [19]:
with mlflow.start_run(run_name="LogisticRegression_Pipeline_v1"):
    
    X = df.drop(columns=["isFraud"])
    y = df["isFraud"]
    
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )
    
    pipeline.fit(X_train, y_train)
    
    y_train_pred = pipeline.predict_proba(X_train)[:, 1]
    y_val_pred = pipeline.predict_proba(X_val)[:, 1]
    
    train_auc = roc_auc_score(y_train, y_train_pred)
    val_auc = roc_auc_score(y_val, y_val_pred)
    
    mlflow.log_metric("train_auc", train_auc)
    mlflow.log_metric("val_auc", val_auc)
    
    mlflow.sklearn.log_model(pipeline, "pipeline_model")
    
    print(train_auc, val_auc)

2026/05/03 14:06:58 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/03 14:06:59 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


0.8528232202013528 0.8501692321025613
🏃 View run LogisticRegression_Pipeline_v1 at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/0/runs/65e65557dad544c092dc2f09c56a3b40
🧪 View experiment at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/0
